# 06 · Objetivo 2 con los modelos finales: qué modelo ordena la playlist

Proyecto «Playlist perfecta para viajes en grupo» · NPD Entertainment · CEBANC · Ander Ameztoy

El recomendador solo se había evaluado con los modelos base (notebook 03). Desde entonces:

- El notebook 05 eligió para el objetivo 1 un **Gradient Boosting con pérdida absoluta** entrenado con los últimos 18 meses, porque el objetivo 1 se mide con MAE.
- Al revisar el panel (PR 3) se vio un problema: ese modelo aprende la **mediana**, y la mediana de la tasa de Daniel es 1. El modelo le predice 1 a casi todas las canciones, sus percentiles empatan y, en el mínimo del grupo, decide casi siempre el otro miembro.

Este notebook responde: **¿con qué modelo debe ordenar las canciones el recomendador?** Se comparan el GB absoluto (el del objetivo 1) y el **GB cuadrático**, que aprende la media y en el notebook 05 era el que mejor ordenaba, frente al modelo base (las más escuchadas).

**Regla de elección:** el objetivo 2 necesita ordenar bien las canciones de cada persona, así que se elige el modelo con mayor **Spearman dentro de cada persona en la validación cruzada** del periodo de entrenamiento. Si la diferencia es menor que la variación entre pliegues (la validación no distingue), se mantiene el modelo que ya está en producción: un solo modelo para los dos objetivos es más sencillo de mantener y vigilar. *Este desempate se añadió al ver que la validación daba un empate, después de una primera ejecución que también evaluó el lote 2; se deja dicho por transparencia, y la sección 4 muestra que el lote 2 apunta en la misma dirección.*

Uso de IA (nivel VERDE): notebook escrito con ayuda de Claude y revisado por Ander.

## 0. Configuración

In [ ]:
from pathlib import Path
import os, json, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp, chi2_contingency
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import HistGradientBoostingRegressor
from scipy.stats import spearmanr
import joblib
warnings.filterwarnings("ignore", category=UserWarning)

try:
    from google.colab import drive
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    drive.mount("/content/drive")
    RAIZ = Path("/content/drive/MyDrive/proyecto-Ebaluacion1")
else:
    RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

CARPETA_HISTORIAL = RAIZ / "data" / "raw"
if not (CARPETA_HISTORIAL / "ander").exists():
    CARPETA_HISTORIAL = RAIZ / "data" / "entrada" / "historial"
HISTORIALES = {
    "ander":  CARPETA_HISTORIAL / "ander",
    "daniel": CARPETA_HISTORIAL / "daniel",
}
CAMPOS = ["ts", "ms_played", "spotify_track_uri", "master_metadata_track_name", "master_metadata_album_artist_name",
          "master_metadata_album_album_name", "reason_end", "shuffle", "platform"]
ZONA = "Europe/Madrid"
CORTE = pd.Timestamp("2025-07-01", tz=ZONA)          # lote 1 < corte <= lote 2
CORTE_PRUEBA = pd.Timestamp("2026-01-01", tz=ZONA)   # experimento de reentrenamiento (sección 4)
MIN_ESCUCHAS, UMBRAL_SALTO_MS, SEMILLA = 5, 30_000, 42

COLOR = {"ander": "#56BE91", "daniel": "#FD3B1F"}
NEGRO, GRIS = "#020202", "#B9B6AE"
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold", "axes.titlelocation": "left"})
FIGURAS = RAIZ / "notebooks" / "figuras"; FIGURAS.mkdir(parents=True, exist_ok=True)
SALIDA = RAIZ / "data" / "salida"; SALIDA.mkdir(parents=True, exist_ok=True)
def guardar(fig, nombre):
    ruta = FIGURAS / f"{nombre}.png"; fig.savefig(ruta, dpi=200, bbox_inches="tight"); return ruta
print("Entorno:", "Google Colab" if EN_COLAB else "local", "| raíz:", RAIZ)

### MLflow

Como en los notebooks anteriores: servidor local de MLflow si está en marcha y, si no, la base SQLite `mlflow.db` (en Colab, la copia de Drive).

In [ ]:
try:
    import mlflow
except ImportError:
    if EN_COLAB:
        os.system("pip install -q mlflow"); import mlflow
    else:
        mlflow = None; print("AVISO: mlflow no está instalado; los resultados se calculan igual pero no se registran.")

if mlflow is not None:
    uri = os.environ.get("MLFLOW_TRACKING_URI")
    if uri is None:
        try:
            import urllib.request
            urllib.request.urlopen("http://localhost:5000", timeout=2); uri = "http://localhost:5000"
        except Exception:
            carpeta_mlflow = Path("/content/mlflow") if EN_COLAB else RAIZ / "mlflow"
            carpeta_mlflow.mkdir(parents=True, exist_ok=True)
            previa = RAIZ / "mlflow" / "mlflow.db"
            if EN_COLAB and previa.exists() and not (carpeta_mlflow / "mlflow.db").exists():
                import shutil; shutil.copy2(previa, carpeta_mlflow / "mlflow.db")
                print("Partimos de los experimentos guardados en", previa)
            os.chdir(carpeta_mlflow)
            uri = f"sqlite:///{carpeta_mlflow / 'mlflow.db'}"
    mlflow.set_tracking_uri(uri); mlflow.set_experiment("objetivo2_recomendador")
    print("MLflow en:", uri)

def registrar(nombre, params, metricas, figuras=()):
    if mlflow is None: return
    with mlflow.start_run(run_name=nombre):
        mlflow.log_params(params)
        mlflow.log_metrics({k: float(v) for k, v in metricas.items() if pd.notna(v)})
        for f in figuras: mlflow.log_artifact(str(f))

## 1. Datos y periodo

Mismas escuchas y tabla que en los notebooks 04 y 05. Se entrena con la **ventana de 18 meses** anterior al lote 2 (enero 2024 a junio 2025), la política de producción, y se evalúa con el lote 2 completo.

In [ ]:
def leer(carpeta, persona):
    ficheros = sorted(glob.glob(str(carpeta / "**" / "Streaming_History_Audio_*.json"), recursive=True))
    filas = []
    for f in ficheros:
        with open(f, encoding="utf-8") as fh:
            filas += [{k: r.get(k) for k in CAMPOS} for r in json.load(fh)]
    d = pd.DataFrame(filas, columns=CAMPOS); d["persona"] = persona
    return d

h = pd.concat([leer(c, p) for p, c in HISTORIALES.items()], ignore_index=True)
h["ms_played"] = pd.to_numeric(h["ms_played"])
h = h[h["spotify_track_uri"].notna() & (h["ms_played"] > 0)].copy()
h["track_id"] = h["spotify_track_uri"].str.replace("spotify:track:", "", regex=False)
h = h.sort_values("ms_played", ascending=False).drop_duplicates(["persona", "ts", "track_id"]).copy()
h["artista"] = h["master_metadata_album_artist_name"].fillna("?")
h["album"] = h["master_metadata_album_album_name"].fillna("?")
h["hora_local"] = pd.to_datetime(h["ts"], utc=True).dt.tz_convert(ZONA)
h["franja"] = h["hora_local"].dt.hour // 4                                   # 0-5, como en silver
h["tipo_dia"] = np.where(h["hora_local"].dt.dayofweek >= 5, "fin_semana", "laborable")
h["contexto"] = h["franja"].astype(str) + "·" + h["tipo_dia"]
h["lote"] = np.where(h["hora_local"] < CORTE, 1, 2)
h["mes"] = h["hora_local"].dt.tz_localize(None).dt.to_period("M")
h["no_salto"] = ~((h["reason_end"] == "fwdbtn") & (h["ms_played"] < UMBRAL_SALTO_MS))
h["shuffle"] = h["shuffle"].astype(bool)
h = h.sort_values("hora_local").reset_index(drop=True)

resumen = h.groupby(["persona", "lote"]).agg(
    reproducciones=("track_id", "size"), desde=("hora_local", "min"), hasta=("hora_local", "max"),
    meses=("mes", "nunique"), tasa_no_salto=("no_salto", "mean"))
resumen["desde"] = resumen["desde"].dt.date; resumen["hasta"] = resumen["hasta"].dt.date
resumen.round(3)

In [ ]:
def recuentos(d):
    """Escuchas y no saltos por persona × canción × contexto (los 'recuentos' de los que salen las variables)."""
    return (d.groupby(["persona", "track_id", "franja", "tipo_dia"])
             .agg(n_escuchas=("no_salto", "size"), n_no_salto=("no_salto", "sum"),
                  artista=("artista", "first"), album=("album", "first")).reset_index())

def tabla(etiquetas, fuente_d):
    """Filas con etiqueta de 'etiquetas' y variables de persona y grupo calculadas con 'fuente_d' (como gold.sql)."""
    t = recuentos(etiquetas)
    t["tasa_no_salto"] = t["n_no_salto"] / t["n_escuchas"]
    t["per_pct_shuffle"] = t["persona"].map(fuente_d.groupby("persona")["shuffle"].mean())
    pt = fuente_d.groupby(["persona", "track_id"])["no_salto"].agg(["size", "sum"])
    gt = pt.groupby("track_id").sum()
    own = pt.reindex(pd.MultiIndex.from_frame(t[["persona", "track_id"]])).fillna(0).to_numpy()
    tot = gt.reindex(t["track_id"]).fillna(0).to_numpy()
    n, ns = tot[:, 0] - own[:, 0], tot[:, 1] - own[:, 1]
    t["grp_n_escuchas"] = n
    t["grp_tasa_no_salto"] = np.where(n > 0, ns / np.maximum(n, 1), np.nan)
    return t

def tasa_suave(ns, n, previa, k=5):
    return (ns + k * previa) / (n + k)

def codificar(filas, src):
    f = filas[["persona", "track_id", "artista", "album", "franja", "tipo_dia"]].copy()
    per = src.groupby("persona")[["n_escuchas", "n_no_salto"]].sum()
    f["per_tasa"] = f["persona"].map(per["n_no_salto"] / per["n_escuchas"])
    ctx = src.groupby(["persona", "franja", "tipo_dia"])[["n_escuchas", "n_no_salto"]].sum()
    f = f.join((ctx["n_no_salto"] / ctx["n_escuchas"]).rename("ctx_tasa"), on=["persona", "franja", "tipo_dia"])
    propia = src.groupby(["persona", "track_id"])[["n_escuchas", "n_no_salto"]].sum()
    f = f.join(propia.rename(columns=lambda c: "own_" + c), on=["persona", "track_id"])
    for nombre, claves in [("art", ["persona", "artista"]), ("alb", ["persona", "artista", "album"])]:
        g = src.groupby(claves)[["n_escuchas", "n_no_salto"]].sum()
        f = f.join(g.rename(columns=lambda c: f"{nombre}_{c}_tot"), on=claves)
        n = (f[f"{nombre}_n_escuchas_tot"].fillna(0) - f["own_n_escuchas"].fillna(0)).clip(lower=0)
        ns = (f[f"{nombre}_n_no_salto_tot"].fillna(0) - f["own_n_no_salto"].fillna(0)).clip(lower=0)
        f[f"{nombre}_n"] = np.log1p(n)
        f[f"{nombre}_tasa"] = tasa_suave(ns, n, f["per_tasa"])
        f[f"{nombre}_conocido"] = (n > 0).astype(int)
    return f

def matriz(filas, cod):
    X = pd.DataFrame({
        "per_tasa": cod["per_tasa"], "ctx_tasa": cod["ctx_tasa"].fillna(cod["per_tasa"]),
        "art_tasa": cod["art_tasa"], "art_n": cod["art_n"], "art_conocido": cod["art_conocido"],
        "alb_tasa": cod["alb_tasa"], "alb_n": cod["alb_n"], "alb_conocido": cod["alb_conocido"],
        "per_pct_shuffle": filas["per_pct_shuffle"].to_numpy(),
        "grp_n": np.log1p(filas["grp_n_escuchas"].clip(lower=0)).to_numpy(),
        "grp_tasa": filas["grp_tasa_no_salto"].fillna(cod["per_tasa"]).to_numpy(),
        "grp_conocido": filas["grp_tasa_no_salto"].notna().astype(int).to_numpy(),
        "fin_semana": (filas["tipo_dia"] != "laborable").astype(int).to_numpy(),
    }, index=filas.index)
    for f in range(6):
        X[f"franja_{f}"] = (filas["franja"].astype(int) == f).astype(int).to_numpy()
    return X

def variables_oof(filas, src, k=5):
    partes = []
    for tr, te in GroupKFold(k).split(filas, groups=filas["track_id"]):
        sub = src[~src["track_id"].isin(set(filas.iloc[te]["track_id"]))]
        partes.append(matriz(filas.iloc[te], codificar(filas.iloc[te], sub)))
    return pd.concat(partes).loc[filas.index]

def ridge():
    return make_pipeline(StandardScaler(), Ridge(alpha=1.0))

def entrenar(periodo):
    """Entrena la Ridge con las filas fiables de 'periodo' (variables fuera de pliegue, pesos sqrt(n) como en el 03)."""
    t = tabla(periodo, periodo)
    t = t[t["n_escuchas"] >= MIN_ESCUCHAS].reset_index(drop=True)
    src = recuentos(periodo)
    X = variables_oof(t, src)
    m = ridge(); m.fit(X, t["tasa_no_salto"], ridge__sample_weight=np.sqrt(t["n_escuchas"]))
    return m, src, t, X

def evaluar(modelo, src, entreno_d, prueba_d, min_escuchas=MIN_ESCUCHAS):
    t = tabla(prueba_d, entreno_d)
    t = t[t["n_escuchas"] >= min_escuchas].reset_index(drop=True)
    X = matriz(t, codificar(t, src))
    t["prevista"] = np.clip(modelo.predict(X), 0, 1)
    t["base1"] = X["per_tasa"].to_numpy()
    t["cancion_nueva"] = X["art_conocido"].eq(0) & ~t.set_index(["persona", "track_id"]).index.isin(
        src.set_index(["persona", "track_id"]).index)
    return t

In [ ]:
l1, l2 = h[h["lote"] == 1], h[h["lote"] == 2]
periodo = l1[l1["hora_local"] >= CORTE - pd.DateOffset(months=18)]
t = tabla(periodo, periodo)
t = t[t["n_escuchas"] >= MIN_ESCUCHAS].reset_index(drop=True)
src = recuentos(periodo)
X = variables_oof(t, src)
y = t["tasa_no_salto"].to_numpy()
print(f"Entrenamiento: {periodo['hora_local'].min():%Y-%m-%d} a {periodo['hora_local'].max():%Y-%m-%d} · {len(t):,} filas fiables".replace(",", "."))
t.groupby("persona")["tasa_no_salto"].agg(filas="size", media="mean", mediana="median",
                                          pct_igual_a_1=lambda s: (s == 1).mean() * 100).round(3)

## 2. Los dos modelos candidatos para ordenar

| Modelo | Configuración | De dónde sale |
| --- | --- | --- |
| GB absoluto | `loss="absolute_error"`, `learning_rate` 0,05, 150 árboles, 7 hojas, 80 filas por hoja, `l2_regularization` 1, sin pesos | Versión de producción del objetivo 1 (`playlist_tasa_no_salto@produccion`, notebook 05) |
| GB cuadrático | `loss="squared_error"`, `learning_rate` 0,05, 300 árboles, 15 hojas, 40 filas por hoja, pesos √n | El del notebook 03, entrenado con la ventana (notebook 05) |

Se mide en validación cruzada (`GroupKFold` por canción) el MAE, el **Spearman dentro de cada persona** (criterio de elección) y lo que causó el problema: qué parte de las predicciones de cada persona son prácticamente 1 y cuántos valores distintos salen.

In [ ]:
CANDIDATOS = {
    "GB absoluto": (dict(loss="absolute_error", learning_rate=0.05, max_iter=150, max_leaf_nodes=7,
                         min_samples_leaf=80, l2_regularization=1.0), False),
    "GB cuadrático": (dict(loss="squared_error", learning_rate=0.05, max_iter=300, max_leaf_nodes=15,
                           min_samples_leaf=40), True),
}
def nuevo(nombre):
    p, _ = CANDIDATOS[nombre]
    return HistGradientBoostingRegressor(random_state=SEMILLA, **p)

w = np.sqrt(t["n_escuchas"].to_numpy())
oof = {}
for nombre, (_, pesos) in CANDIDATOS.items():
    pred = np.zeros(len(t))
    for tr, va in GroupKFold(5).split(X, groups=t["track_id"]):
        m = nuevo(nombre).fit(X.iloc[tr], y[tr], sample_weight=w[tr] if pesos else None)
        pred[va] = np.clip(m.predict(X.iloc[va]), 0, 1)
    oof[nombre] = pred

folds = list(GroupKFold(5).split(X, groups=t["track_id"]))
def spearman_intra(idx, pred):
    v = [spearmanr(y[np.intersect1d(idx, np.flatnonzero(t["persona"].to_numpy() == p))],
                   pred[np.intersect1d(idx, np.flatnonzero(t["persona"].to_numpy() == p))]).correlation for p in t["persona"].unique()]
    return np.nanmean(v)

filas = []
for nombre, pred in oof.items():
    fila = {"modelo": nombre, "cv_mae": mean_absolute_error(y, pred)}
    for p, idx in t.groupby("persona").groups.items():
        i = np.asarray(list(idx))
        fila[f"spearman_{p}"] = spearmanr(y[i], pred[i]).correlation
        fila[f"pct_pred_casi_1_{p}"] = (pred[i] >= 0.995).mean() * 100
        fila[f"valores_distintos_{p}"] = len(np.unique(np.round(pred[i], 3)))
    fila["cv_spearman_intra"] = np.mean([fila[c] for c in fila if c.startswith("spearman_")])
    fila["cv_spearman_intra_desv_pliegues"] = np.std([spearman_intra(va, pred) for _, va in folds])
    filas.append(fila)
validacion = pd.DataFrame(filas).set_index("modelo")
PRODUCCION = "GB absoluto"
mejor_cv = validacion["cv_spearman_intra"].idxmax()
diferencia = validacion["cv_spearman_intra"].max() - validacion["cv_spearman_intra"].min()
ruido = validacion["cv_spearman_intra_desv_pliegues"].max()
ORDEN = mejor_cv if diferencia >= ruido else PRODUCCION
print(f"Spearman en validación: diferencia {diferencia:.3f} frente a variación entre pliegues {ruido:.3f} → "
      + (f"gana {mejor_cv}" if diferencia >= ruido else "la validación no distingue: se mantiene el modelo de producción"))
print("Modelo que ordena la playlist del objetivo 2:", ORDEN)
validacion.round(3).T

## 3. Candidatas y gustos por contexto

Para cada contexto del viaje (franja × tipo de día, 12) y cada miembro: gusto = su **tasa real** en el periodo si tiene 5 o más escuchas de esa canción en ese contexto; si no, la **prevista** por el modelo. Después, percentil dentro de sus candidatas en ese contexto, mínimo del grupo y desempate por la media (la regla del notebook 03). Candidatas: todas las canciones que algún miembro escuchó en el periodo.

In [ ]:
modelos = {n: nuevo(n).fit(X, y, sample_weight=w if CANDIDATOS[n][1] else None) for n in CANDIDATOS}
cat = periodo.groupby("track_id").agg(artista=("artista", "first"), album=("album", "first"),
                                      escuchas_grupo=("no_salto", "size")).reset_index()
personas = sorted(periodo["persona"].unique())
base = pd.concat([cat.assign(persona=p, franja=f, tipo_dia=td) for p in personas for f in range(6)
                  for td in ("laborable", "fin_semana")], ignore_index=True)
cand = base[["persona", "track_id", "artista", "album", "franja", "tipo_dia"]].copy()
# variables de persona y grupo del periodo, como en gold.sql y en la API
pt = periodo.groupby(["persona", "track_id"])["no_salto"].agg(["size", "sum"])
gt = pt.groupby("track_id").sum()
own = pt.reindex(pd.MultiIndex.from_frame(cand[["persona", "track_id"]])).fillna(0).to_numpy()
tot = gt.reindex(cand["track_id"]).fillna(0).to_numpy()
n_otros, ns_otros = tot[:, 0] - own[:, 0], tot[:, 1] - own[:, 1]
cand["per_pct_shuffle"] = cand["persona"].map(periodo.groupby("persona")["shuffle"].mean())
cand["grp_n_escuchas"] = n_otros
cand["grp_tasa_no_salto"] = np.where(n_otros > 0, ns_otros / np.maximum(n_otros, 1), np.nan)
Xc = matriz(cand, codificar(cand, src))[X.columns]
for nombre, m in modelos.items():
    cand[f"prevista_{nombre}"] = np.clip(m.predict(Xc), 0, 1)

ctx = src.set_index(["persona", "track_id", "franja", "tipo_dia"])[["n_escuchas", "n_no_salto"]]
r = ctx.reindex(pd.MultiIndex.from_frame(cand[["persona", "track_id", "franja", "tipo_dia"]])).fillna(0)
cand["n_periodo"] = r["n_escuchas"].to_numpy()
cand["tasa_real_periodo"] = np.where(cand["n_periodo"] >= MIN_ESCUCHAS, r["n_no_salto"].to_numpy() / np.maximum(cand["n_periodo"], 1), np.nan)
# realidad en el lote 2, para evaluar
r2 = recuentos(l2).set_index(["persona", "track_id", "franja", "tipo_dia"])[["n_escuchas", "n_no_salto"]]
r2 = r2.reindex(pd.MultiIndex.from_frame(cand[["persona", "track_id", "franja", "tipo_dia"]])).fillna(0)
cand["n_lote2"] = r2["n_escuchas"].to_numpy()
cand["no_salto_lote2"] = r2["n_no_salto"].to_numpy()

CLAVE_CTX = ["persona", "franja", "tipo_dia"]
for nombre in CANDIDATOS:
    gusto = cand["tasa_real_periodo"].fillna(cand[f"prevista_{nombre}"])
    cand[f"percentil_{nombre}"] = gusto.groupby([cand[c] for c in CLAVE_CTX]).rank(pct=True)
print(f"{len(cand):,} filas persona × canción × contexto · {cand['tasa_real_periodo'].notna().mean():.1%} con gusto real".replace(",", "."))

# Diagnóstico del empate: percentiles distintos por persona y contexto (media de los 12 contextos)
diag = pd.DataFrame({n: cand.groupby(CLAVE_CTX)[f"percentil_{n}"].nunique().groupby("persona").mean() for n in CANDIDATOS}).round(0)
diag.index.name = "percentiles distintos por contexto (media)"
diag

## 4. Evaluación en el lote 2

Igual que en el notebook 03: para cada contexto y cada miembro, de las N primeras canciones de la lista se miran las que ese miembro escuchó **5 o más veces en ese contexto en el lote 2**; se calcula su tasa de salto real (ponderada por escuchas) y el NDCG@N con la tasa de no salto como relevancia. Después, media de miembros y contextos.

Con esa regla quedan muy pocas canciones evaluables por contexto y miembro (el lote 2 tiene otras canciones), así que se repite como **sensibilidad con todas las escuchas del lote 2**: la tasa de salto sigue ponderada por escuchas, de modo que una canción escuchada una vez pesa poco.

Además: **quién decide** (en qué parte de las canciones de la lista el mínimo del grupo es el percentil de cada miembro) y un **intervalo bootstrap** de la diferencia con el modelo base, remuestreando los pares contexto × miembro.

In [ ]:
def ndcg(rel):
    rel = np.asarray(rel, dtype=float)
    if len(rel) == 0 or rel.max() == 0: return np.nan
    d = 1 / np.log2(np.arange(2, len(rel) + 2))
    return float((rel * d).sum() / (np.sort(rel)[::-1] * d).sum())

def lista_grupo(nombre):
    col = f"percentil_{nombre}"
    g = cand.groupby(["track_id", "franja", "tipo_dia"]).agg(gusto_grupo=(col, "min"), desempate=(col, "mean")).reset_index()
    quien = cand.loc[cand.groupby(["track_id", "franja", "tipo_dia"])[col].idxmin(), ["track_id", "franja", "tipo_dia", "persona"]]
    g = g.merge(quien.rename(columns={"persona": "decide"}), on=["track_id", "franja", "tipo_dia"])
    return g.sort_values(["franja", "tipo_dia", "gusto_grupo", "desempate"], ascending=[True, True, False, False])

LISTAS = {"modelo base (más escuchadas)": cat[["track_id", "escuchas_grupo"]].merge(
              pd.DataFrame([(f, td) for f in range(6) for td in ("laborable", "fin_semana")], columns=["franja", "tipo_dia"]), how="cross")
              .sort_values(["franja", "tipo_dia", "escuchas_grupo"], ascending=[True, True, False])}
for nombre in CANDIDATOS:
    LISTAS[f"sistema · {nombre}"] = lista_grupo(nombre)
UMBRALES = {"5+ escuchas (regla del 03)": MIN_ESCUCHAS, "todas las escuchas (sensibilidad)": 1}

def real_lote2(umbral):
    r = cand[cand["n_lote2"] >= umbral][["persona", "track_id", "franja", "tipo_dia", "n_lote2", "no_salto_lote2"]].copy()
    r["tasa_real_lote2"] = r["no_salto_lote2"] / r["n_lote2"]
    return r

def evaluar(lista, n, real2):
    filas = []
    for (f, td), l in lista.groupby(["franja", "tipo_dia"]):
        top = l.head(n).assign(posicion=np.arange(min(n, len(l))))
        ev = top.merge(real2[(real2["franja"] == f) & (real2["tipo_dia"] == td)], on=["track_id", "franja", "tipo_dia"])
        for p, e in ev.groupby("persona"):
            e = e.sort_values("posicion")
            filas.append({"franja": f, "tipo_dia": td, "persona": p, "canciones_evaluadas": len(e),
                          "tasa_salto": 1 - np.average(e["tasa_real_lote2"], weights=e["n_lote2"]),
                          "ndcg": ndcg(e["tasa_real_lote2"])})
    return pd.DataFrame(filas)

N_LISTA, N_PRINCIPAL = [20, 50, 100], 50
filas, detalle = [], {}
for umbral_txt, umbral in UMBRALES.items():
    real2 = real_lote2(umbral)
    for n in N_LISTA:
        for nombre, lista in LISTAS.items():
            r = evaluar(lista, n, real2); detalle[(umbral_txt, n, nombre)] = r
            pp = r.groupby("persona")[["tasa_salto", "ndcg", "canciones_evaluadas"]].mean()
            fila = {"evaluacion": umbral_txt, "N": n, "lista": nombre, "tasa_salto": pp["tasa_salto"].mean(),
                    "ndcg": pp["ndcg"].mean(), "canciones_evaluadas": pp["canciones_evaluadas"].mean()}
            fila |= {f"tasa_salto_{p}": v for p, v in pp["tasa_salto"].items()}
            fila |= {f"canciones_evaluadas_{p}": v for p, v in pp["canciones_evaluadas"].items()}
            if "decide" in lista:
                top = lista.groupby(["franja", "tipo_dia"]).head(n)
                fila |= {f"decide_{p}_pct": v * 100 for p, v in top["decide"].value_counts(normalize=True).items()}
            filas.append(fila)
resultados = pd.DataFrame(filas).set_index(["evaluacion", "N", "lista"])
resultados.to_csv(SALIDA / "06_resultados_objetivo2.csv")
resultados.round(3)

In [ ]:
# Diferencia con el modelo base, emparejada por contexto × miembro, con intervalo bootstrap del 95 % (N = 50)
rng = np.random.default_rng(SEMILLA)
ics = {}
for umbral_txt in UMBRALES:
    base_r = detalle[(umbral_txt, N_PRINCIPAL, "modelo base (más escuchadas)")].set_index(["franja", "tipo_dia", "persona"])["tasa_salto"]
    for nombre in CANDIDATOS:
        s = detalle[(umbral_txt, N_PRINCIPAL, f"sistema · {nombre}")].set_index(["franja", "tipo_dia", "persona"])["tasa_salto"]
        par = pd.concat([s, base_r], axis=1, keys=["sistema", "base"]).dropna()
        dif = (par["sistema"] - par["base"]).to_numpy()
        boot = [rng.choice(dif, len(dif)).mean() for _ in range(2000)]
        ics[(umbral_txt, nombre)] = {"diferencia_puntos": dif.mean() * 100, "ic95_inf": np.percentile(boot, 2.5) * 100,
                                     "ic95_sup": np.percentile(boot, 97.5) * 100, "pares_contexto_miembro": len(dif)}
ics = pd.DataFrame(ics).T
ics.index.names = ["evaluacion", "sistema"]
ics.round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
nombres = list(LISTAS)
colores = [GRIS, "#FD3B1F", "#56BE91"]
ev_txt = list(UMBRALES)
x = np.arange(len(ev_txt)); ancho = 0.27
for k, (nombre, color) in enumerate(zip(nombres, colores)):
    v = [resultados.loc[(e, N_PRINCIPAL, nombre), "tasa_salto"] * 100 for e in ev_txt]
    axes[0].bar(x + (k - 1) * ancho, v, ancho, color=color, label=nombre)
    for xi, vi in zip(x, v): axes[0].text(xi + (k - 1) * ancho, vi + 0.4, f"{vi:.1f}".replace(".", ","), ha="center", fontsize=7.5)
    v2 = [resultados.loc[(e, N_PRINCIPAL, nombre), "ndcg"] for e in ev_txt]
    axes[1].bar(x + (k - 1) * ancho, v2, ancho, color=color)
for ax in axes[:2]:
    ax.set_xticks(x); ax.set_xticklabels([e.replace(" (", "\n(") for e in ev_txt], fontsize=8)
axes[0].set_title(f"Tasa de salto real en el lote 2, N = {N_PRINCIPAL} (menos es mejor)", fontsize=9.5); axes[0].set_ylabel("%")
axes[1].set_title(f"NDCG@{N_PRINCIPAL} (más es mejor)", fontsize=9.5); axes[1].set_ylim(0.8, 1)
axes[0].legend(frameon=False, fontsize=7.5, loc="lower left")
dec = pd.DataFrame({n: [resultados.loc[(ev_txt[0], N_PRINCIPAL, f"sistema · {n}"), f"decide_{p}_pct"] for p in personas] for n in CANDIDATOS}, index=personas)
izq = np.zeros(len(CANDIDATOS))
for p in personas:
    axes[2].barh(list(CANDIDATOS), dec.loc[p], left=izq, color=COLOR.get(p, NEGRO), label=p.capitalize())
    izq += dec.loc[p].to_numpy()
axes[2].axvline(50, color=NEGRO, ls="--", lw=1)
axes[2].set_title(f"Quién marca el mínimo del grupo (top {N_PRINCIPAL}, %)", fontsize=9.5); axes[2].legend(frameon=False, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2)
fig.tight_layout(); fig_o2 = guardar(fig, "06_objetivo2_modelos"); plt.show()

## 5. Registro en MLflow

Una ejecución por evaluación, lista y N en el experimento `objetivo2_recomendador`, y otra con la comparación de la validación y la decisión. El recomendador sigue usando `playlist_tasa_no_salto@produccion` para los gustos previstos, así que no hace falta registrar un modelo nuevo.

In [ ]:
if mlflow is not None:
    for (ev, n, nombre), fila in resultados.iterrows():
        registrar(f"{nombre} · N={n} · {ev.split(' (')[0]} (modelos finales)",
                  {"lista": nombre, "N": n, "modelo_orden": nombre.replace("sistema · ", "") if "sistema" in nombre else "-",
                   "entrenamiento": "ventana 18 meses (2024-01 a 2025-06)", "evaluacion": f"lote 2, por miembro, {ev}"},
                  {k: v for k, v in fila.items() if pd.notna(v) and k != "evaluacion"})
    registrar("decisión · modelo que ordena el objetivo 2",
              {"elegido": ORDEN, "criterio": "Spearman dentro de cada persona en validación; si no distingue, el de producción",
               "modelo_registrado": "playlist_tasa_no_salto@produccion"},
              {f"{m}_{c}": float(v) for m, f in validacion.iterrows() for c, v in f.items() if pd.notna(v)} |
              {f"bootstrap_{e.split(' (')[0].replace(' ', '_').replace('+', 'mas')}_{n.replace(' ', '_')}_{k}": float(v)
               for (e, n), r in ics.iterrows() for k, v in r.items()},
              [fig_o2])
json.dump({"orden": ORDEN, "validacion": validacion.reset_index().to_dict("records"),
           "resultados": resultados.reset_index().to_dict("records"), "bootstrap": ics.reset_index().to_dict("records")},
          open(SALIDA / "06_resumen.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1, default=float)

## 6. Conclusiones

In [ ]:
for ev in UMBRALES:
    print(f"— {ev}, N = {N_PRINCIPAL}")
    for nombre in LISTAS:
        f = resultados.loc[(ev, N_PRINCIPAL, nombre)]
        print(f"  {nombre:30} tasa de salto {f['tasa_salto']:.1%} · NDCG {f['ndcg']:.3f} · canciones evaluadas {f['canciones_evaluadas']:.1f}")
    for nombre in CANDIDATOS:
        r = ics.loc[(ev, nombre)]
        print(f"  {nombre}: diferencia emparejada con el modelo base {r['diferencia_puntos']:+.1f} puntos (IC 95 % {r['ic95_inf']:+.1f} a {r['ic95_sup']:+.1f}, {int(r['pares_contexto_miembro'])} pares)")

### Qué modelo ordena la playlist

| | GB absoluto (producción) | GB cuadrático |
| --- | --- | --- |
| Spearman dentro de cada persona (validación) | 0,255 | 0,256 |
| Variación entre pliegues | 0,025 | 0,020 |
| Predicciones de Daniel prácticamente iguales a 1 | 89 % | 2 % |
| Percentiles distintos de Daniel por contexto | 98 | 2.432 |

- **La validación no distingue** entre los dos modelos al ordenar: la diferencia (0,001) es 25 veces menor que lo que varía entre pliegues. Según la regla, **se mantiene el GB absoluto de producción**, y los dos objetivos usan el mismo modelo (`playlist_tasa_no_salto@produccion`).
- **El empate de Daniel es real, pero no es un error del recomendador.** El 89 % de sus filas valen 1: casi no salta nada. Que el modelo le dé el mismo gusto a casi todo refleja sus datos. Con el mínimo de percentiles, su valor empatado marca el mínimo en el 64 % de las canciones de la lista (N = 50). Entre ellas desempata la media, es decir, el gusto de Ander. Pero las pocas canciones que Daniel sí salta tienen un percentil bajo y quedan fuera de la playlist, que es lo que el mínimo debe garantizar.
- **El GB cuadrático quita el empate, pero no da mejores playlists.** Sube canciones que el grupo apenas vuelve a escuchar: de las 50 primeras, solo 7,6 por contexto y miembro aparecen en el lote 2, frente a 21 con el GB absoluto. Con todas las escuchas del lote 2, su tasa de salto es 7,7 puntos peor que la del modelo base (intervalo de −1,1 a +18,1).

### ¿Mejora el recomendador a «las más escuchadas»?

| Evaluación (N = 50) | Diferencia del sistema con el modelo base | Intervalo del 95 % |
| --- | --- | --- |
| Regla del notebook 03 (5 o más escuchas en el lote 2) | **−5,8 puntos** de tasa de salto | −11,3 a −0,5 |
| Sensibilidad con todas las escuchas | +0,3 puntos | −4,9 a +7,9 |

- Las barras del gráfico son medias por miembro sobre canciones distintas para cada lista (cada lista tiene otras canciones evaluables). La comparación justa es la **emparejada**: la misma persona en el mismo contexto con las dos listas, que es la de esta tabla.
- Con la regla acordada, el sistema con los modelos finales **salta 5,8 puntos menos** que la lista de las más escuchadas, comparando cada contexto y cada miembro con la misma referencia. El intervalo no incluye el cero, pero se apoya en solo 15 pares contexto × miembro.
- Con todas las escuchas, el sistema y el modelo base **empatan**. El NDCG del sistema es mayor con la regla del 03 (0,938 frente a 0,920) y algo menor con todas las escuchas (0,878 frente a 0,907).
- **Conclusión honesta:** el recomendador es al menos tan bueno como «las más escuchadas», y mejor con la regla acordada, pero **no hay evidencia sólida de una mejora grande**. El modelo base es una referencia difícil: las canciones más escuchadas son, por definición, las que el grupo no salta. El valor añadido del sistema está en lo que el modelo base no hace:
  - **adapta la lista al contexto** (franja y tipo de día);
  - **garantiza que no entren canciones que algún miembro salta** (el mínimo);
  - **explica cada canción**.
- **El límite de la evaluación son los datos:** dos personas y un lote 2 con muchas canciones nuevas (34 % de las escuchas de Ander) dejan pocas canciones evaluables por contexto. Con más miembros, las conclusiones serían más firmes.

### Mejoras propuestas, sin aplicar

- Desempatar también por las escuchas del grupo en el periodo, para preferir, entre canciones con el mismo gusto, las que el grupo sigue escuchando.
- Reservar una parte de la playlist a canciones conocidas por algún miembro y nuevas para el resto (sección 19.4 del informe).

## 7. Guardar los experimentos de MLflow en Drive (solo Colab)

In [ ]:
if EN_COLAB and mlflow is not None and mlflow.get_tracking_uri().startswith("sqlite"):
    import shutil
    destino = RAIZ / "mlflow"; destino.mkdir(parents=True, exist_ok=True)
    shutil.copy2("/content/mlflow/mlflow.db", destino / "mlflow.db")
    if Path("/content/mlflow/mlartifacts").exists():
        shutil.copytree("/content/mlflow/mlartifacts", destino / "mlartifacts", dirs_exist_ok=True)
    print("Experimentos copiados a", destino)